# inkcheck / IRP in 5 minutes (CPU, Colab-friendly)

1. Install inkcheck.
2. Check a prediction against a reference ink map: **PASS**, with tile-bootstrap CIs. By default this uses a synthetic stroke pair, because the repo ships no scroll data (see "Data and licensing" in the README). Set `REAL = True` to fetch the real Paris 4 example from the public Vesuvius Challenge bucket instead (the reference is ~4 MB; rebuilding the prediction needs villa, see `examples/fetch_paris4_example.py`).
3. Run a negative control (reference tiles deranged and rotated), which must **FAIL**.
4. Build an IRP card from the shipped case files (no data needed).

Everything runs on CPU in about 2 minutes. In Colab, run the first cell; locally, run `pip install -e .` from the repo instead.

In [ ]:
import os, subprocess, sys
if not os.path.exists("pyproject.toml"):          # Colab / fresh machine
    if not os.path.exists("vesuvius-inkcheck"):
        subprocess.run(["git", "clone", "-q", "https://github.com/dlescanogithub/vesuvius-inkcheck"], check=True)
    os.chdir("vesuvius-inkcheck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
os.environ.setdefault("OMP_NUM_THREADS", "2")

## 1. A check: synthetic pair (default) or Paris 4 (held-out segment 20231005123336, public `dense_native` 9 µm checkpoint)

- The prediction is a 9.6 µm ink map; the reference is at 19.2 µm.
- Synthetic: 768² prediction px starting at reference pixel (128, 128).
- Paris 4: `paris4_dn_pred_9.6um.png` (1536² px, 14.7 mm) starting at pixel (256, 256) of a 1280² crop of the published new_canon map.

In [ ]:
import json
from PIL import Image
from inkcheck import CheckConfig, run_check
from inkcheck.io import load_map
REAL = False                                   # True: fetch Paris 4 (needs VILLA_PY / VILLA_SRC for the prediction)
if REAL:
    subprocess.run([sys.executable, "examples/fetch_paris4_example.py", "--with-pred"], check=True)
    PRED, REF, ORIGIN = "examples/data/paris4_dn_pred_9.6um.png", "examples/data/paris4_ref_crop_19.2um.png", (256, 256)
else:
    subprocess.run([sys.executable, "examples/make_synthetic_example.py"], check=True)
    PRED, REF, ORIGIN = "examples/data/synthetic_pred_9.6um.png", "examples/data/synthetic_ref_19.2um.png", (128, 128)
pred, ref = load_map(PRED), load_map(REF)
cfg = CheckConfig(pred_um=9.6, ref_um=19.2, ref_origin=ORIGIN, workers=2)
res = run_check(pred, ref, cfg, log=lambda s: None)
print(res.verdict, "|", res.summary)
b = res.numbers["bootstrap"]
print("r_bp 95% tile-bootstrap CI:", [round(x, 3) for x in b["r_ci"]])
print("stroke recall - chance:", round(b["recall_minus_chance"], 2), [round(x, 2) for x in b["recall_minus_chance_ci"]])

`r_bp` is the band-pass correlation after registration. Its p-value comes from 192 shifted placements of the reference (the search-matched null), not from the CI.

The CI is a 2 mm-tile bootstrap that holds the registration fixed. It measures how precise the number is, not whether it is significant. See the negative controls in the README.

PASS needs all four of: p_r ≤ 0.01, p_stroke ≤ 0.05, r_bp ≥ 0.20 and stroke recall ≥ 0.50.

In [ ]:
subprocess.run(["inkcheck", PRED, REF, "--pred-um", "9.6", "--ref-um", "19.2", "--ref-origin", f"{ORIGIN[0]},{ORIGIN[1]}",
                "--workers", "2", "--out", "out/check"])
Image.open("out/check.png")

## 2. Negative control (must FAIL)

- Synthetic: a prediction made from unrelated strokes (`synthetic_negative_pred_9.6um.png`).
- Paris 4 (`REAL = True`): the same prediction against the reference with its 2 mm tiles permuted so that no tile stays in place, each rotated by 90° or 270°.

A plain shuffle keeps the horizontal text-line layout. On Paris 4 it came out **WEAK**, not FAIL. That is why IRP requires stroke recall for PASS: r alone can be explained by line layout.

In [ ]:
import numpy as np
if not REAL:
    r2 = run_check(load_map("examples/data/synthetic_negative_pred_9.6um.png"), ref, cfg, log=lambda s: None)
else:
    win = ref[256:256 + 768, 256:256 + 768].copy(); T = 104; n = 768 // T          # 104 px = 2 mm at 19.2 um
    tiles = [win[i*T:(i+1)*T, j*T:(j+1)*T].copy() for i in range(n) for j in range(n)]
    rng = np.random.default_rng(1)
    while True:
        perm = rng.permutation(len(tiles))
        if not (perm == np.arange(len(tiles))).any(): break
    neg = win.copy()
    for k, p in enumerate(perm):
        i, j = divmod(k, n); neg[i*T:(i+1)*T, j*T:(j+1)*T] = np.rot90(tiles[p], int(rng.choice([1, 3])))
    r2 = run_check(pred, neg, CheckConfig(pred_um=9.6, ref_um=19.2, ref_origin=(0, 0), workers=2), log=lambda s: None)
print(r2.verdict, "|", r2.summary)

## 3. An IRP card from the shipped case files

- Node A asks whether held-out models see known ink.
- Node B asks whether the scan is good enough.
- Node C asks whether a second scan or another reference agrees.
- Node D gives the decision: what to do with this segment.

In [ ]:
subprocess.run(["inkcheck", "report", "examples/irp/paris4.case.json", "--out", "out/cards"], check=True)
print(open("out/cards/paris4.md").read()[:1500])
Image.open("out/cards/paris4.png")

## 4. Your own model

```bash
inkcheck MY_PRED.tif PUBLISHED_REF.png --pred-um 9.6 --ref-um 2.4 --ref-origin Y,X --mask valid.png --out out/mine
```

- If you run [inkbench](https://github.com/inksweep/inkbench), `inkcheck inkbench $INKBENCH_RESULTS` prints its ranking with leakage flags.
- `--case-out` turns the inkbench maps of one segment into an IRP case.

inkbench ranks models; IRP decides what to do with a segment.